# Lecture 8 Lab — Spindle Thermal Drift Compensation
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn, PyTorch |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = xxx  # replace with 3 last digits of your student ID.

## 1. The Problem

A machining centre's spindle heats up as it runs. The front and rear bearings, the motor windings and the column casting all expand — at different rates and with different time constants — and the net effect is that the tool tip drifts away from the position the controller believes it is at.

On a precision job with a ±5 µm tolerance, a drift of 40–80 µm over a warm-up period is the difference between a good part and scrap.

**Your task:** build a model that predicts the current tip displacement from temperature sensors and operating state, accurately enough that the controller could apply an equal-and-opposite offset. This is *thermal error compensation*, a standard feature on high-end machine tools.

### Learning objectives

By the end of this lab, you should be able to:

- Define a regression network by subclassing a module, using a reusable building block rather than a flat stack.
- Explain and demonstrate why the choice of loss function can matter more than the choice of architecture.
- Train with a three-way split, monitor validation loss, and apply early stopping with a patience setting.
- Save a trained model *and everything needed to reproduce its inputs*, then reload and verify it.
- Report regression error in physical units and judge it against an engineering tolerance.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__)

## 3. Part A — Simulating the Machine

Each *session* is one run of the spindle: an ambient temperature, a speed profile with occasional step changes, and 150 minutes of minute-by-minute readings. Four thermal states evolve with first-order lags and different time constants; tip displacement is a mildly nonlinear function of how far each has risen above ambient.

Two deliberate imperfections, both realistic:

- **Sensor noise** on every temperature channel.
- **Probe glitches**: roughly 1.5% of *displacement* readings are badly wrong — a chip, a splash of coolant, an EMI spike on the capacitive probe.

The dataframe also carries `true_displacement_um` and `is_glitched`. **These are reference columns for evaluation only.** As in Week 5, nothing in Parts B–G may use them.

In [ ]:
def simulate_spindle_sessions(n_sessions=120, minutes=150, seed=0, probe_glitch_rate=0.015):
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_sessions):
        ambient = rng.uniform(17.0, 27.0)
        speed_profile = np.full(minutes, rng.uniform(2000, 12000))
        for _ in range(rng.integers(0, 3)):           # occasional speed step
            t0 = rng.integers(10, minutes - 10)
            speed_profile[t0:] = rng.uniform(2000, 12000)

        T_front = T_rear = T_motor = T_col = ambient
        for t in range(minutes):
            spd = speed_profile[t]
            heat = (spd / 10000.0) ** 1.5             # heat input is nonlinear in speed

            # first-order thermal lags, different time constants (minutes)
            T_front += (ambient + 22.0 * heat - T_front) / 25.0
            T_rear  += (ambient + 14.0 * heat - T_rear)  / 38.0
            T_motor += (ambient + 30.0 * heat - T_motor) / 12.0
            T_col   += (ambient +  9.0 * heat - T_col)   / 70.0

            d_front, d_motor, d_col = T_front - ambient, T_motor - ambient, T_col - ambient

            # bearings and motor push the tip out; column growth pulls the frame back
            disp = (1.45 * d_front + 0.55 * d_motor - 0.80 * d_col
                    + 0.045 * d_front ** 1.6 + 0.010 * d_front * d_motor)
            disp_true = disp + rng.normal(0, 0.35)

            disp_measured, glitched = disp_true, False
            if rng.random() < probe_glitch_rate:
                disp_measured = disp_true + rng.choice([-1, 1]) * rng.uniform(10, 25)
                glitched = True

            rows.append({
                "T_front_bearing": T_front + rng.normal(0, 0.12),
                "T_rear_bearing":  T_rear  + rng.normal(0, 0.12),
                "T_motor":         T_motor + rng.normal(0, 0.15),
                "T_column":        T_col   + rng.normal(0, 0.10),
                "T_ambient":       ambient + rng.normal(0, 0.08),
                "spindle_speed_rpm": spd,
                "runtime_min": float(t),
                "tip_displacement_um": disp_measured,
                "true_displacement_um": disp_true,   # evaluation only
                "is_glitched": glitched,             # evaluation only
            })
    return pd.DataFrame(rows)


df = simulate_spindle_sessions(seed = s_id)
FEATURES = ["T_front_bearing", "T_rear_bearing", "T_motor", "T_column",
            "T_ambient", "spindle_speed_rpm", "runtime_min"]
TARGET = "tip_displacement_um"
print("Student ID : "+str(s_id))
print(f"{len(df)} readings from 120 sessions")
print(f"Glitched readings: {int(df['is_glitched'].sum())} ({100*df['is_glitched'].mean():.1f}%)")
df[FEATURES + [TARGET]].describe().round(2)

In [ ]:
# One session's warm-up curve: temperatures rising, displacement following
print("Student ID : "+str(s_id))
session = df.iloc[:150]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
for col, c in [("T_front_bearing", "#FF6A39"), ("T_motor", "#6B4F9C"),
               ("T_rear_bearing", "#3E5C76"), ("T_column", "#8A97A8")]:
    axes[0].plot(session["runtime_min"], session[col], label=col, linewidth=1.6)
axes[0].set_ylabel("Temperature (°C)"); axes[0].legend(fontsize=8); axes[0].set_title("One warm-up session")
axes[1].plot(session["runtime_min"], session["tip_displacement_um"], color="#1B2A41", linewidth=1.4)
axes[1].set_ylabel("Tip displacement (µm)"); axes[1].set_xlabel("Runtime (min)")
axes[1].axhline(5, color="#FF6A39", linestyle="--", linewidth=1, label="±5 µm tolerance")
axes[1].axhline(-5, color="#FF6A39", linestyle="--", linewidth=1)
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

> Note how fast the drift leaves the tolerance band, and note the occasional spikes in the lower trace — those are the probe glitches. Keep them in mind; they drive the most important result in this lab.

## 4. Part B — Three-Way Split and Scaling

Weeks 6 and 7 used train/test. Early stopping needs a third split: something to *watch* during training that is neither trained on nor reserved for the final verdict.

We split by **session**, not by row. Readings one minute apart are almost identical, so a random row split would put near-duplicates on both sides and inflate every score — a subtle form of leakage worth recognising.

TODO : implement the 3-way data split to training, validation, test

In [ ]:
df["session_id"] = np.arange(len(df)) // 150

sessions = df["session_id"].unique()
# TODO : split data in sessions to training, validation, test
# the resulting data portion must be named sess_train, sess_val, sess_test, respectively, 
# to conform with the code that follows.
?
?

tr = df[df.session_id.isin(sess_train)] # training dataset
va = df[df.session_id.isin(sess_val)]  # validation dataset
te = df[df.session_id.isin(sess_test)] # test dataset
print(f"Sessions  -> train {len(sess_train)}, val {len(sess_val)}, test {len(sess_test)}")
print(f"Readings  -> train {len(tr)}, val {len(va)}, test {len(te)}")

scaler = StandardScaler().fit(tr[FEATURES].values)          # fit on TRAIN only
Xtr = scaler.transform(tr[FEATURES].values).astype(np.float32)
Xva = scaler.transform(va[FEATURES].values).astype(np.float32)
Xte = scaler.transform(te[FEATURES].values).astype(np.float32)

ytr = tr[TARGET].values.astype(np.float32)
yva = va[TARGET].values.astype(np.float32)
yte = te[TARGET].values.astype(np.float32)
print("Student ID : "+str(s_id))
# reference values, for evaluation only
yte_true = te["true_displacement_um"].values.astype(np.float32)
te_clean = ~te["is_glitched"].values
print(f"Clean test readings (used for honest scoring): {te_clean.sum()} of {len(te)}")

In [ ]:
def to_loader(X, y, batch_size=128, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))   # note the unsqueeze
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_dl = to_loader(Xtr, ytr, shuffle=True)
val_dl = to_loader(Xva, yva)

xb, yb = next(iter(train_dl))
print("Student ID : "+str(s_id))
print("batch features", xb.shape, " batch target", yb.shape)

> The `unsqueeze` gives the target shape `(batch, 1)` to match the network's output. If you leave it as `(batch,)`, the loss silently broadcasts into a `batch × batch` matrix — no error is raised, and training quietly produces nonsense. This is the single nastiest bug in PyTorch regression.

## 5. Part C — A Custom Regression Module

Fill in the two `TODO`s. The point of `DenseBlock` is reuse: define the repeated pattern once, then stack it. When you want to add dropout or change the activation in Week 9, you change one class rather than six lines.

In [ ]:
class DenseBlock(nn.Module):
    """One linear layer followed by an activation -- the repeated unit of our network."""
    def __init__(self, n_in, n_out):
        super().__init__()
        # TODO: create self.linear (a Linear layer from n_in to n_out)
        #       and self.act (a ReLU activation).
        self.linear = None
        self.act = None
    def forward(self, x):
        return self.act(self.linear(x))


class ThermalDriftNet(nn.Module):
    """Predicts tip displacement (um) from 7 sensor/state features."""
    def __init__(self, n_features, hidden=(64, 32)):
        super().__init__()
        blocks, n_in = [], n_features
        for h in hidden:
            blocks.append(DenseBlock(n_in, h))
            n_in = h
        self.blocks = nn.Sequential(*blocks)
        # TODO: create self.head -- a Linear layer from n_in to 1 output.
        #       No activation here: the target is an unbounded physical quantity.
        self.head = None

    def forward(self, x):
        return self.head(self.blocks(x))

print("Student ID : "+str(s_id))
model = ThermalDriftNet(len(FEATURES))
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Part D — A Reusable Training Function with Early Stopping

Rather than write the loop twice, wrap it. The function trains a fresh model, tracks validation loss each epoch, keeps the best weights it has seen, and stops once validation has failed to improve for `patience` epochs.

TODO : Implement the missing steps in train_model() below. 

In [ ]:
import copy
print("Student ID : "+str(s_id))
def train_model(model, loss_fn, train_dl, val_dl, n_epochs=200, lr=0.01, patience=15, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    best_val, best_state, best_epoch, waited = float("inf"), None, 0, 0

    for epoch in range(n_epochs):
        model.train()
        batch_losses = []
        for xb, yb in train_dl:
            # TODO : implement the missing steps
            ? 
            loss = loss_fn(?,?)
            ?
            ?
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_losses = [loss_fn(model(xb), yb).item() for xb, yb in val_dl]

        history["train"].append(np.mean(batch_losses))
        history["val"].append(np.mean(val_losses))

        if history["val"][-1] < best_val - 1e-5:
            best_val = history["val"][-1]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch, waited = epoch, 0
        else:
            waited += 1
            if waited >= patience:
                if verbose:
                    print(f"  early stop at epoch {epoch+1}; best was epoch {best_epoch+1}")
                break

    model.load_state_dict(best_state)          # restore the best weights, not the last
    return model, history, best_epoch

> Restoring `best_state` at the end is what makes this early stopping rather than just "stopping early". Without it you keep the weights from the *worst* epoch of your patience window.

## 7. Part E — MSE versus Huber, Head to Head

Now the central experiment. Same architecture, same data, same optimizer, same seed — only the loss differs.

Fill in the `TODO`: create a Huber loss. Use `delta=2.0`, meaning errors beyond 2 µm are treated linearly rather than quadratically.

In [ ]:
torch.manual_seed(0)
model_mse, hist_mse, ep_mse = train_model(
    ThermalDriftNet(len(FEATURES)), nn.MSELoss(), train_dl, val_dl)
print(f"MSE model trained (best epoch {ep_mse+1})")
print("Student ID : "+str(s_id))

In [ ]:
# TODO: create a Huber loss function with delta=2.0 and assign it to huber_loss.
huber_loss = None

print("Student ID : "+str(s_id))
torch.manual_seed(0)
model_huber, hist_huber, ep_huber = train_model(
    ThermalDriftNet(len(FEATURES)), huber_loss, train_dl, val_dl)
print(f"Huber model trained (best epoch {ep_huber+1})")

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).squeeze(1).numpy()

def rmse_um(preds):
    """RMSE against the TRUE displacement, on clean (non-glitched) test readings only."""
    return float(np.sqrt(mean_squared_error(yte_true[te_clean], preds[te_clean])))

p_mse = predict(model_mse, Xte)
p_huber = predict(model_huber, Xte)
print("Student ID : "+str(s_id))
print(f"Trained with MSE   : RMSE {rmse_um(p_mse):5.3f} um   R2 {r2_score(yte_true[te_clean], p_mse[te_clean]):.4f}")
print(f"Trained with Huber : RMSE {rmse_um(p_huber):5.3f} um   R2 {r2_score(yte_true[te_clean], p_huber[te_clean]):.4f}")
print(f"\nImprovement from changing the loss alone: {100*(rmse_um(p_mse)-rmse_um(p_huber))/rmse_um(p_mse):.1f}%")

In [ ]:
print("Student ID : "+str(s_id))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (h, name) in zip(axes, [(hist_mse, "MSE"), (hist_huber, "Huber")]):
    ax.plot(h["train"], color="#3E5C76", label="train")
    ax.plot(h["val"], color="#FF6A39", label="validation")
    ax.set_title(f"{name} loss"); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
axes[0].set_ylabel("loss")
plt.tight_layout(); plt.show()

> The two loss curves are on different scales and cannot be compared to each other directly — Huber and MSE measure "wrong" differently. What *is* comparable is the RMSE in µm printed above, because that is measured the same way for both.

## 8. Part F — Is It Good Enough? Compare Against a Baseline

A number is meaningless without a comparison and a requirement. Two of each:

TODO: implement a linear regression model as a baseline for comparison.

In [ ]:
# TODO : create a linear regression model as a baseline, using scikit-learn
# and fit on Xtr, ytr, then predict on Xte
lin = None
p_lin = None

print("Student ID : "+str(s_id))
results = pd.DataFrame({
    "RMSE_um": [rmse_um(p_lin), rmse_um(p_mse), rmse_um(p_huber)],
    "R2": [r2_score(yte_true[te_clean], p_lin[te_clean]),
           r2_score(yte_true[te_clean], p_mse[te_clean]),
           r2_score(yte_true[te_clean], p_huber[te_clean])],
}, index=["Linear regression", "Neural net (MSE)", "Neural net (Huber)"])
print(results.round(4))

uncompensated = float(np.sqrt(np.mean(yte_true[te_clean] ** 2)))
print(f"\nUncompensated drift (RMS): {uncompensated:.1f} um")
print(f"Tolerance band           : +/- 5.0 um")

In [ ]:
print("Student ID : "+str(s_id))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(results.index, results["RMSE_um"], color=["#8A97A8", "#3E5C76", "#FF6A39"])
axes[0].axhline(5.0, color="#6B4F9C", linestyle="--", linewidth=1.5, label="±5 µm tolerance")
axes[0].set_ylabel("Residual RMSE (µm)"); axes[0].set_title("Residual error after compensation")
axes[0].tick_params(axis="x", rotation=15); axes[0].legend(fontsize=8)

axes[1].scatter(yte_true[te_clean], p_huber[te_clean], s=6, alpha=0.4, color="#FF6A39")
lims = [yte_true[te_clean].min(), yte_true[te_clean].max()]
axes[1].plot(lims, lims, color="#1B2A41", linewidth=1.2, linestyle="--")
axes[1].set_xlabel("True displacement (µm)"); axes[1].set_ylabel("Predicted (µm)")
axes[1].set_title("Huber model: predicted vs. actual")
plt.tight_layout(); plt.show()

## 9. Part G — Save the Model the Way You Would Ship It

A trained model that exists only in this session is not a deliverable. Save the weights, the architecture arguments needed to rebuild the object, **and the scaler statistics** — a model trained on scaled inputs will produce confident nonsense if fed raw °C.

In [ ]:
checkpoint = {
    "state_dict": model_huber.state_dict(),
    "n_features": len(FEATURES),
    "hidden": (64, 32),
    "feature_names": FEATURES,
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "target_units": "micrometres",
    "loss": "Huber(delta=2.0)",
}
torch.save(checkpoint, "thermal_drift_model.pt")
print("Student ID : "+str(s_id))
print("Saved thermal_drift_model.pt")

In [ ]:
# Reload from scratch and verify the predictions are identical
ckpt = torch.load("thermal_drift_model.pt", weights_only=False)

reloaded = ThermalDriftNet(ckpt["n_features"], hidden=ckpt["hidden"])
reloaded.load_state_dict(ckpt["state_dict"])
print("Student ID : "+str(s_id))
p_reloaded = predict(reloaded, Xte)
max_diff = float(np.max(np.abs(p_reloaded - p_huber)))
print(f"Largest disagreement between original and reloaded model: {max_diff:.2e} um")
assert max_diff < 1e-5, "Reloaded model does not reproduce the original predictions!"
print("Reload verified.")

In [ ]:
# What the controller would actually do: raw sensor reading -> compensation offset
def predict_from_raw(raw_readings, ckpt, model):
    """raw_readings: dict of feature name -> raw sensor value (deg C, rpm, min)"""
    x = np.array([[raw_readings[f] for f in ckpt["feature_names"]]], dtype=np.float32)
    x = (x - ckpt["scaler_mean"]) / ckpt["scaler_scale"]      # the scaler is essential
    model.eval()
    with torch.no_grad():
        return float(model(torch.tensor(x, dtype=torch.float32)).item())

example = {"T_front_bearing": 42.0, "T_rear_bearing": 33.0, "T_motor": 51.0,
           "T_column": 27.5, "T_ambient": 21.0, "spindle_speed_rpm": 9000.0,
           "runtime_min": 90.0}
offset = predict_from_raw(example, ckpt, reloaded)
print("Student ID : "+str(s_id))
print(f"Predicted drift : {offset:+.2f} um")
print(f"Controller applies compensation offset: {-offset:+.2f} um")

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. Changing only the loss function produced a large drop in error. Explain the mechanism — what does squaring the error do to a reading that is 20 µm wrong, and why does that distort the fitted model?**

*Your answer:*

**2. Look at the R² column in Part F. Linear regression scores very highly there, yet its RMSE in µm is noticeably worse than the network's. Explain how both facts can be true at once, and say which number you would put in front of a production engineer.**

*Your answer:*

**3. We split by session rather than by individual reading. Explain what would have gone wrong with a random row-level split, and name the Week 6 concept it relates to.**

*Your answer:*

**4. The saved checkpoint includes the scaler's mean and scale. Describe concretely what would happen on the machine if an engineer deployed the weights but rebuilt the scaler from that week's production data instead.**

*Your answer:*

**5. Early stopping selected the epoch with the lowest *validation* loss, and we then reported performance on the *test* set. Explain why reporting the best validation loss instead would have been dishonest.**

*Your answer:*

## 11. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The warm-up session plot (§3), both loss curves (§7), and the comparison/scatter figures (§8).
- The reload verification output from §9.
- Written answers to the five reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| `DenseBlock` and `ThermalDriftNet` correctly defined | 20% |
| Huber loss correctly applied; MSE/Huber comparison produced and interpreted | 25% |
| Baseline comparison and error judged against the µm tolerance | 20% |
| Model saved with scaler, reloaded, and verified | 15% |
| Reflection questions | 10% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

Each output cell that shows incorrect s_id printout will get -1 penalty per cell.

---
**Next week:** *Convolutional Neural Networks* — raw images instead of hand-engineered features, applied to automated visual inspection.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>